# Dodatek 10.2: Použití nástrojů

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příklad hřiště](#example-playground)

## Nastavení

Spusťte následující buňku nastavení pro načtení vašeho API klíče a vytvoření pomocné funkce `get_completion`.

In [ ]:
```python
# Přepsáno pro volání Claude 3 Sonnet, který je obecně lepší při používání nástrojů, a zahrnuje stop_sequences
# Import vestavěné knihovny regulárních výrazů v Pythonu
import re
import boto3
import json

# Import modulu hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Přepsání proměnné MODEL_NAME v úložišti IPython pro použití Sonnet místo modelu Haiku
MODEL_NAME='anthropic.claude-3-sonnet-20240229-v1:0'
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(messages, system_prompt="", prefill="", stop_sequences=None):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "temperature": 0.0,
            "top_p": 1,
            "messages":messages,
            "system": system_prompt,
            "stop_sequences": stop_sequences
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')
```

---

## Lekce

I když se to může na první pohled zdát konceptuálně složité, používání nástrojů, neboli volání funkcí, je ve skutečnosti docela jednoduché! Už znáte všechny dovednosti potřebné k implementaci používání nástrojů, což je ve skutečnosti jen kombinace substituce a řetězení promptů.

V předchozích cvičeních se substitucí jsme nahrazovali text v promptech. Při používání nástrojů nahrazujeme výsledky nástrojů nebo funkcí v promptech. Claude nemůže doslova volat nebo přistupovat k nástrojům a funkcím. Místo toho máme Claude:
1. Vypíše název nástroje a argumenty, které chce volat
2. Zastaví jakoukoli další generaci odpovědí, zatímco je nástroj volán
3. Poté znovu zadáme prompt s připojenými výsledky nástroje

Volání funkcí je užitečné, protože rozšiřuje schopnosti Claude a umožňuje mu zvládat mnohem složitější, vícekrokové úkoly.  
Některé příklady funkcí, které můžete Claude poskytnout:
- Kalkulačka
- Počítadlo slov
- Dotazování do SQL databáze a získávání dat
- Weather API

Můžete přimět Claudea k použití nástrojů kombinací těchto dvou prvků:

1. Systémový prompt, ve kterém poskytneme Claudeovi vysvětlení konceptu použití nástrojů a také podrobný popisný seznam nástrojů, ke kterým má přístup
2. Řídící logika, pomocí které lze organizovat a provádět požadavky Claudea na použití nástrojů

### Plán použití nástrojů

*Tato lekce učí náš současný formát použití nástrojů. Nicméně, v blízké budoucnosti budeme aktualizovat a zlepšovat funkčnost použití nástrojů, včetně:*
* *Efektivnějšího formátu pro definice a volání funkcí*
* *Robustnějšího zpracování chyb a pokrytí okrajových případů*
* *Těsnější integrace se zbytkem našeho API*
* *Lepší spolehlivosti a výkonu, zejména pro složitější úkoly použití nástrojů*

### Příklady

Pro povolení použití nástrojů v Claude začínáme systémovým promptem. V tomto speciálním systémovém promptu pro použití nástrojů říkáme Claude:
* Základní premisu použití nástrojů a co to obnáší
* Jak může Claude volat a používat nástroje, které mu byly poskytnuty
* Podrobný seznam nástrojů, ke kterým má přístup v tomto konkrétním scénáři

Zde je první část systémového promptu, která vysvětluje použití nástrojů Claude. Tato část systémového promptu je zobecnitelná pro všechny instance, kdy Claude žádáme o použití nástrojů. Struktura volání nástrojů, kterou Claude poskytujeme (`<function_calls> [...] </function_calls>`), je struktura, kterou byl Claude speciálně vyškolen používat, takže doporučujeme, abyste se jí drželi.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

Zde je druhá část systémového promptu, která definuje přesné nástroje, ke kterým má Claude přístup v této konkrétní situaci. V tomto příkladu poskytneme Claudovi nástroj kalkulačky, který přijímá tři parametry: dva operandy a operátor.

Poté spojíme obě části systémového promptu.

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

Nyní můžeme dát Claudovi otázku, která vyžaduje použití nástroje `calculator`. Použijeme `<function_calls\>` v `stop_sequences` k detekci, zda a kdy Claude volá funkci.

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Získat odpověď od Claude
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Nyní můžeme extrahovat parametry z volání funkce Claude a skutečně spustit funkci jménem Claude.

Nejprve definujeme kód funkce.

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

Poté extrahujeme parametry z odpovědi funkce Claude. Pokud všechny parametry existují, spustíme nástroj kalkulačky.

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

Nyní, když máme výsledek, musíme tento výsledek správně naformátovat tak, aby když ho předáme zpět Claudovi, Claude pochopil, k jakému nástroji se tento výsledek vztahuje. Existuje pevný formát, který byl Claude naučen rozpoznávat:
```
<function_results>
<result>
<tool_name>{TOOL_NAME}</tool_name>
<stdout>
{TOOL_RESULT}
</stdout>
</result>
</function_results>
```

Spusťte buňku níže, abyste naformátovali výše uvedený výsledek nástroje do této struktury.

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

Nyní stačí tento výsledek poslat zpět Claudovi připojením výsledku ke stejnému řetězci zpráv jako předtím, a máme hotovo!

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Sestavení celé konverzace
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Vytisknout odpověď od Claude
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- KONEČNÝ VÝSLEDEK -------------")
print(final_response)

Gratulujeme k úspěšnému spuštění celého řetězce nástrojů od začátku do konce!

Co když teď dáme Claudovi otázku, která vůbec nevyžaduje použití daného nástroje?

In [ ]:
```python
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Získat odpověď od Claude
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)
```

Úspěch! Jak můžete vidět, Claude věděl, že nemá volat funkci, když to nebylo potřeba.

Pokud byste chtěli experimentovat s prompty lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

## Cvičení
- [Cvičení 10.2.1 - SQL](#exercise-1021---SQL)

### Cvičení 10.2.1 - SQL
V tomto cvičení budete psát prompt pro dotazování a zápis do nejmenší "databáze" na světě. Zde je inicializovaná databáze, která je ve skutečnosti jen slovníkem.

In [ ]:
db = {
    "users": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
        {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
    ],
    "products": [
        {"id": 1, "name": "Widget", "price": 9.99},
        {"id": 2, "name": "Gadget", "price": 14.99},
        {"id": 3, "name": "Doohickey", "price": 19.99}
    ]
}

A zde je kód pro funkce, které zapisují do databáze a čtou z ní.

In [ ]:
def get_user(user_id):
    for user in db["users"]:
        if user["id"] == user_id:
            return user
    return None

def get_product(product_id):
    for product in db["products"]:
        if product["id"] == product_id:
            return product
    return None

def add_user(name, email):
    user_id = len(db["users"]) + 1
    user = {"id": user_id, "name": name, "email": email}
    db["users"].append(user)
    return user

def add_product(name, price):
    product_id = len(db["products"]) + 1
    product = {"id": product_id, "name": name, "price": price}
    db["products"].append(product)
    return product

K vyřešení cvičení začněte definováním systémového promptu jako `system_prompt_tools_specific_tools` výše. Ujistěte se, že zahrnete název a popis každého nástroje, spolu s názvem, typem a popisem každého parametru pro každou funkci. Níže jsme vám poskytli základní strukturu.

In [ ]:
system_prompt_tools_specific_tools_sql = """
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools_sql

Až budete připraveni, můžete vyzkoušet svůj systémový prompt pro definici nástroje na níže uvedených příkladech. Stačí spustit níže uvedenou buňku!

In [ ]:
```python
examples = [
    "Přidej uživatele do databáze jménem Deborah.",
    "Přidej produkt do databáze jménem Thingo",
    "Řekni mi jméno Uživatele 2",
    "Řekni mi jméno Produktu 3"
]

for example in examples:
    message = {
        "role": "user",
        "content": example
    }

    # Získat a vytisknout odpověď od Claude
    function_calling_response = get_completion([message], system_prompt=system_prompt, stop_sequences=stop_sequences)
    print(example, "\n----------\n\n", function_calling_response, "\n*********\n*********\n*********\n\n")
```

Pokud jste to udělali správně, zprávy volající funkce by měly správně volat funkce `add_user`, `add_product`, `get_user` a `get_product`.

Pro extra body přidejte nějaké buňky s kódem a napište kód pro zpracování parametrů. Poté zavolejte funkce s parametry, které vám Claude poskytne, abyste viděli stav "databáze" po volání.

❓ Pokud chcete vidět možné řešení, spusťte buňku níže!

In [ ]:
print(hints.exercise_10_2_1_solution)

### Gratulujeme!

Gratulujeme k naučení používání nástrojů a volání funkcí! Pokud se chcete dozvědět více o vyhledávání a RAG, přejděte do poslední části přílohy.

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Získat odpověď od Claude
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Sestavení celé konverzace
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Výpis odpovědi od Claude
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- KONEČNÝ VÝSLEDEK -------------")
print(final_response)

In [ ]:
```python
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Získat odpověď od Claude
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)
```